# Importar codigos

Se obtienen las funciones del codigo

In [2]:
import os
import pandas as pd
import numpy as np
from PIL import Image
import subprocess
import math
import random



In [3]:
import import_ipynb
from Recodificacion import *

In [4]:
import import_ipynb
from Codigos import *
from Codigos_obj import calcular_ms_ssim_lpips

In [ ]:
METODOS      = ["pillow", "ffmpeg", "imagemagick"]
FORMATOS     = ["jpg", "webp", "png"]
CARPETA_RAIZ = "imagenes"
CALIDADES_POOL = [10, 30, 50, 70, 90]
CALIDADES_POR_IMAGEN = 3
SEED = 42  # reproducible — mismas calidades siempre para la misma imagen
LPIPS_MIN_SIZE = 161

# Funcion para obtener todos los valores cuantitativos de la imagen

Obtiene los valores seleccionados de la parte teorica

In [ ]:
import os

def recodificar_imagen(
    img_path,
    metodo,
    formato,
    calidad
):

    # -----------------------------------------
    # VALIDACIONES
    # -----------------------------------------

    metodo = metodo.lower()
    formato = formato.lower()

    metodos_validos = [
        "pillow",
        "ffmpeg",
        "imagemagick"
    ]

    formatos_validos = [
        "jpg",
        "png",
        "webp"
    ]

    if metodo not in metodos_validos:
        raise ValueError(
            f"Método no soportado: {metodo}"
        )

    if formato not in formatos_validos:
        raise ValueError(
            f"Formato no soportado: {formato}"
        )

    # -----------------------------------------
    # RUTA SALIDA
    # -----------------------------------------

    nombre = os.path.splitext(
        os.path.basename(img_path)
    )[0]

    out_dir = os.path.join(
        "imagenes",
        metodo,
        formato
    )

    os.makedirs(out_dir, exist_ok=True)

    out_path = os.path.join(
        out_dir,
        f"{nombre}_{metodo}_q{calidad}.{formato}"
    )

    # -----------------------------------------
    # LLAMAR FUNCIÓN CORRESPONDIENTE
    # -----------------------------------------

    if metodo == "pillow":

        convert_image_pillow(
            input_path=img_path,
            output_path=out_path,
            quality=calidad
        )

    elif metodo == "ffmpeg":

        convert_with_ffmpeg(
            input_path=img_path,
            output_path=out_path,
            quality=calidad
        )

    elif metodo == "imagemagick":

        convert_with_imagemagick(
            input_path=img_path,
            output_path=out_path,
            quality=calidad
        )

    # -----------------------------------------
    # VALIDAR GENERACIÓN
    # -----------------------------------------

    if not os.path.exists(out_path):

        raise RuntimeError(
            f"No se generó el archivo:\n{out_path}"
        )

    # -----------------------------------------
    # RETORNO
    # -----------------------------------------

    return out_path

In [ ]:
def features_histograma(path, prefix):
    import numpy as np

    f = {}

    hist, hist_norm = obtener_histograma_intensidades(path)

    # guardar todo
    f[f"{prefix}_hist"] = hist.tolist()
    f[f"{prefix}_hist_norm"] = hist_norm.tolist()
    f[f"{prefix}_hist_bins"] = list(range(256))

    # derivados
    hist_norm_safe = hist_norm[hist_norm > 0]

    f[f"{prefix}_hist_entropy"] = -np.sum(hist_norm_safe * np.log2(hist_norm_safe))
    f[f"{prefix}_hist_std"] = np.std(hist_norm)
    f[f"{prefix}_hist_mean"] = np.mean(hist_norm)
    f[f"{prefix}_hist_min"] = np.min(hist_norm)
    f[f"{prefix}_hist_max"] = np.max(hist_norm)

    mu = np.mean(hist_norm)
    f[f"{prefix}_hist_skew"] = np.mean((hist_norm - mu)**3)
    f[f"{prefix}_hist_kurt"] = np.mean((hist_norm - mu)**4)

    return f

In [ ]:
def features_generales(path, prefix):
    f = {}

    # básicas
    f[f"{prefix}_rango"] = obtener_rango_dinamico(path)
    f[f"{prefix}_std"] = obtener_desv_estandar_brillo(path)

    # contraste
    f[f"{prefix}_michelson"] = obtener_contraste_michelson(path)

    # frecuencia
    f[f"{prefix}_hf"] = obtener_energia_altas_frecuencias(path)
    f[f"{prefix}_lf"] = obtener_energia_bajas_frecuencias(path)
    f[f"{prefix}_hlfr"] = obtener_hlfr(path)

    # bordes multi escala
    f[f"{prefix}_edges_s1"] = obtener_densidad_bordes(path, sigma=1.0)
    f[f"{prefix}_edges_s2"] = obtener_densidad_bordes(path, sigma=2.0)
    f[f"{prefix}_edges_s3"] = obtener_densidad_bordes(path, sigma=3.0)

    # gradiente
    f[f"{prefix}_grad_mean"] = obtener_magnitud_media_gradiente(path)
    f[f"{prefix}_grad_std"] = obtener_desv_estandar_gradiente(path)

    # textura
    f[f"{prefix}_glcm_contrast"] = obtener_contraste_textura(path)
    f[f"{prefix}_glcm_hom"] = obtener_homogeneidad_textura(path)
    f[f"{prefix}_glcm_energy"] = obtener_energia_textura(path)
    f[f"{prefix}_glcm_corr"] = obtener_correlacion_glcm(path)

    # color
    f[f"{prefix}_sat_mean"] = obtener_media_saturacion(path)
    f[f"{prefix}_sat_var"] = obtener_varianza_saturacion(path)

    return f

In [ ]:
def extraer_features(path, prefix):
    f = {}

    f.update(features_generales(path, prefix))
    f.update(features_histograma(path, prefix))

    return f

In [ ]:
def procesar_imagen(
    img_path,
    df,
    metodo="pillow",
    formato="jpg",
    calidad=50
):
    import pandas as pd

    if not isinstance(df, pd.DataFrame):
        raise TypeError("df debe ser un pandas.DataFrame")

    # 1. recodificar
    out_path = recodificar_imagen(img_path, metodo, formato, calidad)

    # 2. features
    feats_orig = extraer_features(img_path, "orig")
    feats_comp = extraer_features(out_path, "comp")

    # 3. métrica objetivo
    metricas = calcular_ms_ssim_lpips(img_path, out_path)

    # 4. fila
    fila = {
        "img_original":     img_path,
        "img_recodificada": out_path,
        "metodo":           metodo,
        "formato":          formato,
        "calidad":          calidad,
        **feats_orig,
        **feats_comp,
        **metricas
    }

    return pd.concat([df, pd.DataFrame([fila])], ignore_index=True)

In [ ]:
def procesar_dataset_completo(
    df: pd.DataFrame,
    formato_origen: str = "jpg",
    inicio: int = 0,
    fin: int = None
) -> pd.DataFrame:

    random.seed(SEED)

    formato_orig = formato_origen.lower().strip()

    if formato_orig not in FORMATOS:
        raise ValueError(f"formato_origen debe ser uno de {FORMATOS}, recibido: '{formato_orig}'")

    carpeta = os.path.join(CARPETA_RAIZ, formato_orig)

    if not os.path.isdir(carpeta):
        raise FileNotFoundError(f"Carpeta no encontrada: {carpeta}")

    imagenes = sorted([
        f for f in os.listdir(carpeta)
        if f.lower().endswith(f".{formato_orig}")
    ])

    if not imagenes:
        print(f"⚠ Sin imágenes en: {carpeta}")
        return df

    formatos_salida = [f for f in FORMATOS if f != formato_orig]
    fmt_A = formatos_salida[0]
    fmt_B = formatos_salida[1]

    mitad = math.ceil(len(imagenes) / 2)
    mitad_A = imagenes[:mitad]
    mitad_B = imagenes[mitad:]

    todas = list(zip(
        mitad_A + mitad_B,
        [fmt_A] * len(mitad_A) + [fmt_B] * len(mitad_B)
    ))

    fin_real = fin if fin is not None else len(todas)
    subset   = todas[inicio:fin_real]

    print(f"\n{'='*60}")
    print(f"📁 {carpeta}  ({len(imagenes)} imágenes total)")
    print(f"   Procesando [{inicio}:{fin_real}] → {len(subset)} imágenes")
    print(f"{'='*60}")

    for i in range(inicio):
        random.sample(CALIDADES_POOL, CALIDADES_POR_IMAGEN)

    for idx_global, (nombre_archivo, formato_sal) in enumerate(subset, start=inicio):

        img_path = os.path.join(carpeta, nombre_archivo)
        metodo   = METODOS[idx_global % len(METODOS)]

        calidades_img = random.sample(CALIDADES_POOL, CALIDADES_POR_IMAGEN)

        print(f"\n  🖼 [{idx_global}] {nombre_archivo}  →  {formato_sal}  [{metodo}]  calidades={calidades_img}")

        for calidad in calidades_img:
            try:
                df = procesar_imagen(
                    img_path = img_path,
                    df       = df,
                    metodo   = metodo,
                    formato  = formato_sal,
                    calidad  = calidad
                )
            except Exception as e:
                print(f"    ❌ q={calidad:2d}  {e}")
                continue

    print(f"\n✅ [{inicio}:{fin_real}] completado — filas acumuladas: {len(df)}")

    return df

In [ ]:
import os

def obtener_imagenes_faltantes(carpeta, formato, df):

    print("\n===== DEBUG INICIO =====\n")

    # ---------------------------
    # 1. ARCHIVOS EN CARPETA
    # ---------------------------
    archivos_carpeta = [
        f for f in os.listdir(carpeta)
        if f.lower().endswith(f".{formato.lower()}")
    ]

    print(f"📁 Total en carpeta ({carpeta}): {len(archivos_carpeta)}")
    print("Ejemplos carpeta:")
    for f in archivos_carpeta[:10]:
        print("  ", f)

    # ---------------------------
    # 2. DATAFRAME (nombres)
    # ---------------------------
    nombres_df = [
        os.path.basename(path)
        for path in df["img_original"]
        if isinstance(path, str)
    ]

    print(f"\n📊 Total en DataFrame: {len(nombres_df)}")
    print("Ejemplos DataFrame:")
    for f in nombres_df[:10]:
        print("  ", f)

    # ---------------------------
    # 3. CONJUNTOS
    # ---------------------------
    set_carpeta = set(archivos_carpeta)
    set_df = set(nombres_df)

    print(f"\n🧮 Únicos carpeta: {len(set_carpeta)}")
    print(f"🧮 Únicos dataframe: {len(set_df)}")

    # ---------------------------
    # 4. DIFERENCIA
    # ---------------------------
    faltantes = set_carpeta - set_df

    print(f"\n❌ FALTANTES: {len(faltantes)}")
    print("Ejemplos faltantes:")
    for f in list(faltantes)[:10]:
        print("  ", f)

    print("\n===== DEBUG FIN =====\n")

    return sorted(list(faltantes))

In [ ]:
def reprocesar_faltantes(
    df: pd.DataFrame,
    imagenes_faltantes: list,
    formato_origen: str = "jpg"
) -> pd.DataFrame:

    random.seed(SEED)

    formato_orig = formato_origen.lower().strip()
    carpeta      = os.path.join(CARPETA_RAIZ, formato_orig)
    formatos_salida = [f for f in FORMATOS if f != formato_orig]

    # Recuperar el formato de salida original de cada imagen
    # basado en su posición en el listado completo ordenado
    imagenes_todas = sorted([
        f for f in os.listdir(carpeta)
        if f.lower().endswith(f".{formato_orig}")
    ])

    mitad = math.ceil(len(imagenes_todas) / 2)
    mitad_A = imagenes_todas[:mitad]   # → fmt_A
    mitad_B = imagenes_todas[mitad:]   # → fmt_B

    fmt_A = formatos_salida[0]
    fmt_B = formatos_salida[1]

    print(f"\n{'='*60}")
    print(f"🔁 Reprocesando {len(imagenes_faltantes)} imágenes faltantes [{formato_orig}]")
    print(f"{'='*60}")

    for nombre_archivo in imagenes_faltantes:

        img_path = os.path.join(carpeta, nombre_archivo)

        if not os.path.exists(img_path):
            print(f"  ⚠ No encontrada en disco: {nombre_archivo}")
            continue

        # Recuperar índice y grupo original para mantener consistencia
        if nombre_archivo in mitad_A:
            idx        = mitad_A.index(nombre_archivo)
            formato_sal = fmt_A
        elif nombre_archivo in mitad_B:
            idx        = mitad_B.index(nombre_archivo)
            formato_sal = fmt_B
        else:
            print(f"  ⚠ No encontrada en ningún grupo: {nombre_archivo}")
            continue

        # Mismo método que le correspondía originalmente
        metodo = METODOS[idx % len(METODOS)]

        # Mismas calidades que le correspondían originalmente
        # random.seed + mismo orden garantiza reproducibilidad
        random.seed(SEED)
        for i, img in enumerate(mitad_A + mitad_B):
            calidades = random.sample(CALIDADES_POOL, CALIDADES_POR_IMAGEN)
            if img == nombre_archivo:
                calidades_img = calidades
                break

        print(f"\n  🖼 {nombre_archivo}  →  {formato_sal}  [{metodo}]  calidades={calidades_img}")

        for calidad in calidades_img:
            try:
                df = procesar_imagen(
                    img_path = img_path,
                    df       = df,
                    metodo   = metodo,
                    formato  = formato_sal,
                    calidad  = calidad
                )
                print(f"    ✅ q={calidad}")
            except Exception as e:
                print(f"    ❌ q={calidad}  {e}")
                continue

    print(f"\n✅ Reprocesamiento completado — filas acumuladas: {len(df)}")
    return df